In [0]:
import os, sys, subprocess
REPO = "/Workspace/Users/khanoza1@cibcpte.com/Repos/test"
assert os.path.isdir(f"{REPO}/src/radar_stap"), "repo not readable from this compute (group clusters cannot see your home folder)"
env = {**os.environ, "PYTHONPATH": f"{REPO}/src", "PYTHONDONTWRITEBYTECODE": "1", "OPENBLAS_NUM_THREADS": "4"}
RADAR = [sys.executable, "scripts/radar.py"]
print(subprocess.run(RADAR + ["--help"], cwd=REPO, env=env, capture_output=True, text=True).stdout)

In [0]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests"], cwd=REPO, env=env,
                   capture_output=True, text=True, timeout=1800)
print(r.stdout[-6000:]); print(r.stderr[-3000:])

In [0]:
r = subprocess.run(RADAR + ["e2e-overlook", "--frames", "12", "--no-stap",
                    "--output-dir", "/tmp/e2e_quick"], cwd=REPO, env=env,
                   capture_output=True, text=True, timeout=1800)
print(r.stdout[-6000:]); print(r.stderr[-4000:])

In [0]:
env_full = {**os.environ, "PYTHONPATH": f"{REPO}/src", "PYTHONDONTWRITEBYTECODE": "1", "OPENBLAS_NUM_THREADS": str(os.cpu_count()), "PYTHONUNBUFFERED": "1"}
log = open("/tmp/e2e_full.log", "w")
proc = subprocess.Popen(RADAR + ["e2e-overlook", "--frames", "10",
                         "--output-dir", "/tmp/e2e_full"], cwd=REPO, env=env_full, stdout=log, stderr=subprocess.STDOUT)
print("started pid", proc.pid, "| cpus", os.cpu_count())

In [0]:
print("running" if proc.poll() is None else f"finished rc={proc.returncode}")
print(open("/tmp/e2e_full.log").read()[-6000:])

In [0]:
print(subprocess.run(["ps", "-o", "pid,etime,time,pcpu,rss,cmd", "-p", str(proc.pid)], capture_output=True, text=True).stdout)
for d in ["/tmp/e2e_full", "/tmp/e2e_full/uncalibrated", "/tmp/e2e_full/calibrated"]:
    print(d, sorted(os.listdir(d)) if os.path.exists(d) else "-")